# Stage 2: Brown Shipley Transformation

**Purpose**: Transform Stage 1 raw data into standardized `individual_dfm_consolidated` format

**Replicates Excel Workbook's "Edited" Sheet**:
- Policy mapping (Account Number -> Policyholder_Number)
- Security identifier resolution (SEDOL priority, ISIN fallback, synthetic cash codes)
- Asset name enrichment from security master
- Include/Remove flagging with exclusion reasons
- `Movt`-based tolerance check flags for validation (98-102)
- Decision traceability

**Input**: `stage1_brown_shipley_positions_raw`, `stage1_brown_shipley_cash_raw`

**Output**: `individual_dfm_consolidated` (row-for-row with Stage 1, enriched)

**Row Count Expectation**: Same as Stage 1 raw (e.g., 220 in -> 220 out, with flags)

In [ ]:
# ========== Establish Workspace Context ==========
# CRITICAL: Required when notebook is in a subfolder
# This ensures Fabric's lakehouse context is properly initialized
try:
    spark.sql("SELECT 1")
    print("[INFO] Workspace context initialized")
except Exception as e:
    print(f"[WARNING] Workspace context issue: {str(e)}")
    print("[ACTION] Ensure lakehouse is attached in Fabric notebook UI")

# ========== Parameters ==========
period = "2026-03"            # YYYY-MM
run_id = "manual_test_run"    # passed by orchestrator in real run

try:
    period = mssparkutils.notebook.params.get("period") or period
    run_id = mssparkutils.notebook.params.get("run_id") or run_id
except Exception:
    pass

dfm_id = "brown_shipley"
dfm_name = "Brown Shipley"
profile_id = "brown_shipley_default"

print(f"[STAGE 2] Starting Brown Shipley transformation")
print(f"  Period: {period}")
print(f"  Run ID: {run_id}")
print(f"  DFM: {dfm_name}")

In [ ]:
# ========== Imports ==========
from pyspark.sql import functions as F
from pyspark.sql import Window
from datetime import datetime
import json

print("[STAGE 2] Imports complete")

## Step 1: Read Stage 1 Raw Tables

Read positions and cash data from Stage 1 raw tables (created by v2 ingestion).

In [ ]:
# ========== Read Stage 1 Raw Data ==========
print("\n[STEP 1] Reading Stage 1 raw tables...")
print("=" * 70)

def table_exists(table_name):
    try:
        return spark.catalog.tableExists(table_name)
    except Exception:
        return False

# Read positions (securities)
positions_table = "stage1_brown_shipley_positions_raw"
if not table_exists(positions_table):
    print(f"[ERROR] Table {positions_table} not found. Run nb_ingest_brown_shipley_v2 first.")
    mssparkutils.notebook.exit("FAILED")

positions = (
    spark.table(positions_table)
    .filter((F.col("period") == period) & (F.col("run_id") == run_id))
)
positions_count = positions.count()
print(f"  Positions (securities): {positions_count} rows")

# Read cash
cash_table = "stage1_brown_shipley_cash_raw"
if table_exists(cash_table):
    cash = (
        spark.table(cash_table)
        .filter((F.col("period") == period) & (F.col("run_id") == run_id))
    )
    cash_count = cash.count()
    print(f"  Cash: {cash_count} rows")
else:
    print("  Cash: table not found (will use positions only)")
    cash = None

if positions_count == 0:
    print("[ERROR] No data found for this period/run_id. Check Stage 1 ingestion.")
    mssparkutils.notebook.exit("NO_DATA")

print("\n[STEP 1] Stage 1 data loaded successfully")

## Step 2: Combine Positions + Cash

Full outer join on `client_id` to preserve all rows from both sources.

In [ ]:
# ========== Combine Positions and Cash ==========
print("\n[STEP 2] Combining positions and cash...")
print("=" * 70)

if cash is not None and cash.count() > 0:
    # Detect optional columns so we can build a safe select list.
    pos_has_movt = "movt" in positions.columns
    cash_has_movt = "movt" in cash.columns

    # Full outer join on client_id to preserve all rows
    combined_join = positions.alias("pos").join(
        cash.alias("cash"),
        F.col("pos.client_id") == F.col("cash.client_id"),
        "full_outer"
    )

    # Coalesce common fields
    select_exprs = [
        # Provenance
        F.coalesce(F.col("pos.period"), F.col("cash.period")).alias("period"),
        F.coalesce(F.col("pos.run_id"), F.col("cash.run_id")).alias("run_id"),
        F.coalesce(F.col("pos.dfm_id"), F.col("cash.dfm_id")).alias("dfm_id"),
        F.coalesce(F.col("pos.source_file"), F.col("cash.source_file")).alias("source_file"),
        F.coalesce(F.col("pos.source_row_id"), F.col("cash.source_row_id")).alias("source_row_id"),

        # Policy identifiers
        F.coalesce(F.col("pos.client_id"), F.col("cash.client_id")).alias("client_id"),

        # Dates
        F.coalesce(F.col("pos.value_date"), F.col("cash.value_date")).alias("value_date"),

        # Security identifiers (from positions only)
        F.col("pos.isin_code").alias("isin"),
        F.col("pos.sedol_code").alias("sedol"),
        F.col("pos.description_of_security").alias("instrument_name"),
        F.col("pos.type_of_financial_instrument").alias("instrument_type"),

        # Position values (from positions)
        F.col("pos.balance").alias("balance_local"),
        F.col("pos.accrued_interest").alias("accrued_interest_local"),
        F.col("pos.currency_code").alias("position_currency"),

        # Cash values (from cash)
        F.col("cash.accounting_balance").alias("cash_balance_local"),
        F.col("cash.account_currency").alias("cash_currency")
    ]

    # Optional Movt field used for DFM process Step 7 (98-102 tolerance check).
    if pos_has_movt and cash_has_movt:
        select_exprs.append(F.coalesce(F.col("pos.movt"), F.col("cash.movt")).alias("movt_raw"))
    elif pos_has_movt:
        select_exprs.append(F.col("pos.movt").alias("movt_raw"))
    elif cash_has_movt:
        select_exprs.append(F.col("cash.movt").alias("movt_raw"))
    else:
        select_exprs.append(F.lit(None).cast("string").alias("movt_raw"))

    combined = combined_join.select(*select_exprs)

    print(f"  Movt column source: positions={pos_has_movt}, cash={cash_has_movt}")
else:
    # No cash data - use positions only
    pos_has_movt = "movt" in positions.columns
    combined = positions.select(
        "period", "run_id", "dfm_id", "source_file", "source_row_id",
        F.col("client_id").alias("client_id"),
        F.col("value_date").alias("value_date"),
        F.col("isin_code").alias("isin"),
        F.col("sedol_code").alias("sedol"),
        F.col("description_of_security").alias("instrument_name"),
        F.col("type_of_financial_instrument").alias("instrument_type"),
        F.col("balance").alias("balance_local"),
        F.col("accrued_interest").alias("accrued_interest_local"),
        F.col("currency_code").alias("position_currency"),
        F.lit(None).cast("string").alias("cash_balance_local"),
        F.lit(None).cast("string").alias("cash_currency"),
        F.col("movt").alias("movt_raw") if pos_has_movt else F.lit(None).cast("string").alias("movt_raw")
    )
    print(f"  Movt column source: positions={pos_has_movt}, cash=False")

combined_count = combined.count()
print(f"  Combined rows: {combined_count}")
print(f"  Expected: {positions_count} (row-for-row with Stage 1)")

if combined_count != positions_count:
    print(f"  [WARNING] Row count mismatch. Expected {positions_count}, got {combined_count}")

print("\n[STEP 2] Positions and cash combined successfully")

## Step 3: Load Reference Data

Load policy mappings, security master, and FX rates.

In [ ]:
# ========== Load Reference Data ==========
print("\n[STEP 3] Loading reference data...")
print("=" * 70)

# Policy Mapping
policy_mapping_path = "Files/config/policy_mapping.csv"
try:
    policy_mapping = (
        spark.read.option("header", True).csv(policy_mapping_path)
        .filter(F.col("dfm_id") == dfm_id)
        .select(
            F.col("dfm_policy_ref").alias("client_id_map"),
            F.col("ih_policy_ref").alias("policyholder_number")
        )
    )
    policy_map_count = policy_mapping.count()
    print(f"  Policy mappings: {policy_map_count} rows for {dfm_name}")
except Exception as e:
    print(f"  [WARNING] Could not load policy_mapping.csv: {e}")
    print("  Will use client_id as policyholder_number (no mapping)")
    policy_mapping = None

# Security Master
security_master_path = "Files/config/security_master.csv"
try:
    security_master = (
        spark.read.option("header", True).csv(security_master_path)
        .select(
            F.upper(F.trim(F.col("isin"))).alias("isin_master"),
            F.upper(F.trim(F.col("sedol"))).alias("sedol_master"),
            F.col("asset_name").alias("asset_name_master"),
            F.col("asset_class"),
            F.col("currency_iso")
        )
    )
    security_master_count = security_master.count()
    print(f"  Security master: {security_master_count} rows")
except Exception as e:
    print(f"  [WARNING] Could not load security_master.csv: {e}")
    security_master = None

# FX Rates (prefer managed table from nb_load_fx_rates)
try:
    fx_rates = (
        spark.table("fx_rates")
        .filter(F.col("period") == period)
        .select(
            F.upper(F.trim(F.col("from_currency"))).alias("currency"),
            F.col("rate").cast("double").alias("fx_rate")
        )
        .dropDuplicates(["currency"])
    )
    fx_count = fx_rates.count()
    print(f"  FX rates (table): {fx_count} currencies for period={period}")
except Exception as e:
    print(f"  [WARNING] Could not load fx_rates table for period={period}: {e}")
    print("  Trying fallback file Files/config/fx_rates.csv")
    try:
        fx_rates = (
            spark.read.option("header", True).csv("Files/config/fx_rates.csv")
            .select(
                F.upper(F.trim(F.col("currency"))).alias("currency"),
                F.col("rate_to_gbp").cast("double").alias("fx_rate")
            )
        )
        fx_count = fx_rates.count()
        print(f"  FX rates (csv fallback): {fx_count} currencies")
    except Exception as e2:
        print(f"  [WARNING] Could not load fx_rates.csv fallback: {e2}")
        fx_rates = None

# IH Report bridge (optional): expected at least a policy column; supports optional movt and exclude status.
ih_report_mapping = None
try:
    ih_raw = spark.read.option("header", True).csv("Files/config/ih_report_mapping.csv")

    ih_cols = {c.lower().strip(): c for c in ih_raw.columns}
    policy_col = None
    for candidate in ["policy_number", "policyholder_number", "policy_ref", "client_id", "policy"]:
        if candidate in ih_cols:
            policy_col = ih_cols[candidate]
            break

    movt_col = None
    for candidate in ["movt", "movement", "movt_percent"]:
        if candidate in ih_cols:
            movt_col = ih_cols[candidate]
            break

    status_col = None
    for candidate in ["ih_status", "status", "exclude_flag", "include_exclude", "action"]:
        if candidate in ih_cols:
            status_col = ih_cols[candidate]
            break

    if policy_col is None:
        raise Exception(f"No policy column found in IH report mapping. Available columns: {ih_raw.columns}")

    ih_select = [
        F.upper(F.trim(F.col(policy_col))).alias("ih_policy_number"),
        F.col(movt_col).alias("movt_from_ih") if movt_col is not None else F.lit(None).cast("string").alias("movt_from_ih"),
        F.col(status_col).alias("ih_status_raw") if status_col is not None else F.lit(None).cast("string").alias("ih_status_raw")
    ]

    ih_report_mapping = ih_raw.select(*ih_select)
    ih_count = ih_report_mapping.count()
    print(f"  IH Report mapping (csv): {ih_count} rows")
except Exception:
    try:
        if table_exists("ih_report_mapping"):
            ih_raw = spark.table("ih_report_mapping")
            ih_cols = {c.lower().strip(): c for c in ih_raw.columns}

            policy_col = None
            for candidate in ["policy_number", "policyholder_number", "policy_ref", "client_id", "policy"]:
                if candidate in ih_cols:
                    policy_col = ih_cols[candidate]
                    break

            movt_col = None
            for candidate in ["movt", "movement", "movt_percent"]:
                if candidate in ih_cols:
                    movt_col = ih_cols[candidate]
                    break

            status_col = None
            for candidate in ["ih_status", "status", "exclude_flag", "include_exclude", "action"]:
                if candidate in ih_cols:
                    status_col = ih_cols[candidate]
                    break

            if policy_col is None:
                raise Exception(f"No policy column found in ih_report_mapping table. Available columns: {ih_raw.columns}")

            ih_select = [
                F.upper(F.trim(F.col(policy_col))).alias("ih_policy_number"),
                F.col(movt_col).alias("movt_from_ih") if movt_col is not None else F.lit(None).cast("string").alias("movt_from_ih"),
                F.col(status_col).alias("ih_status_raw") if status_col is not None else F.lit(None).cast("string").alias("ih_status_raw")
            ]

            ih_report_mapping = ih_raw.select(*ih_select)
            ih_count = ih_report_mapping.count()
            print(f"  IH Report mapping (table): {ih_count} rows")
    except Exception as e:
        print(f"  [WARNING] IH Report mapping not available: {e}")

if ih_report_mapping is None:
    print("  [INFO] IH Report mapping not loaded. Include/Remove logic will default to Remove when IH lookup is required.")

print("\n[STEP 3] Reference data loaded")

## Step 4: Policy Mapping

Map `client_id` → `policyholder_number` using policy_mapping reference.

In [ ]:
# ========== Apply Policy Mapping ==========
print("\n[STEP 4] Applying policy mapping...")
print("=" * 70)

if policy_mapping is not None:
    enriched = combined.join(
        policy_mapping,
        combined["client_id"] == policy_mapping["client_id_map"],
        "left"
    )

    # Track mapping decision
    enriched = enriched.withColumn(
        "policy_mapping_applied",
        F.when(F.col("policyholder_number").isNotNull(), F.lit(True)).otherwise(F.lit(False))
    )

    # If no mapping found, use client_id as fallback
    enriched = enriched.withColumn(
        "policyholder_number",
        F.coalesce(F.col("policyholder_number"), F.col("client_id"))
    )

    mapped_count = enriched.filter(F.col("policy_mapping_applied") == True).count()
    print(f"  Rows with policy mapping: {mapped_count} / {combined_count}")
    print(f"  Rows using client_id fallback: {combined_count - mapped_count}")
else:
    # No mapping available - use client_id directly
    enriched = combined.withColumn("policyholder_number", F.col("client_id"))
    enriched = enriched.withColumn("policy_mapping_applied", F.lit(False))
    print(f"  Using client_id as policyholder_number (no mapping table)")

# Optional IH bridge: enrich Movt and include/remove lookup metadata by policy number.
if ih_report_mapping is not None:
    ih_policy_rollup = (
        ih_report_mapping
        .filter(F.col("ih_policy_number").isNotNull() & (F.trim(F.col("ih_policy_number")) != ""))
        .groupBy("ih_policy_number")
        .agg(
            F.count(F.lit(1)).alias("ih_lookup_count"),
            F.max(
                F.when(
                    F.upper(F.trim(F.coalesce(F.col("ih_status_raw"), F.lit("")))) == "EXCLUDE",
                    F.lit(1)
                ).otherwise(F.lit(0))
            ).alias("ih_exclude_flag_int"),
            F.first(F.col("ih_status_raw"), ignorenulls=True).alias("ih_status_raw"),
            F.first(F.col("movt_from_ih"), ignorenulls=True).alias("movt_from_ih")
        )
        .withColumn("ih_exclude_flag", F.col("ih_exclude_flag_int") == 1)
        .drop("ih_exclude_flag_int")
    )

    enriched = enriched.join(
        ih_policy_rollup,
        F.upper(F.trim(enriched["policyholder_number"])) == ih_policy_rollup["ih_policy_number"],
        "left"
    )

    enriched = enriched.withColumn("movt_raw", F.coalesce(F.col("movt_raw"), F.col("movt_from_ih")))
    enriched = enriched.withColumn("ih_lookup_count", F.coalesce(F.col("ih_lookup_count"), F.lit(0)))
    enriched = enriched.withColumn("ih_exclude_flag", F.coalesce(F.col("ih_exclude_flag"), F.lit(False)))
    enriched = enriched.withColumn("ih_policy_exists", F.col("ih_lookup_count") > 0)

    ih_movt_count = enriched.filter(F.col("movt_from_ih").isNotNull()).count()
    ih_match_count = enriched.filter(F.col("ih_policy_exists") == True).count()
    ih_exclude_count = enriched.filter(F.col("ih_exclude_flag") == True).count()

    print(f"  IH policy matches by policy number: {ih_match_count}")
    print(f"  IH Movt backfilled rows: {ih_movt_count}")
    print(f"  IH Exclude rows: {ih_exclude_count}")
else:
    enriched = enriched.withColumn("ih_lookup_count", F.lit(0))
    enriched = enriched.withColumn("ih_exclude_flag", F.lit(False))
    enriched = enriched.withColumn("ih_status_raw", F.lit(None).cast("string"))
    enriched = enriched.withColumn("ih_policy_exists", F.lit(False))
    print("  IH bridge skipped (ih_report_mapping not available)")

print("\n[STEP 4] Policy mapping complete")

## Step 5: Security Identifier Resolution

Priority: SEDOL → ISIN → Synthetic cash code (`TPY_CASH_{CURRENCY}`).

In [ ]:
# ========== Security Identifier Resolution ==========
print("\n[STEP 5] Resolving security identifiers...")
print("=" * 70)

# Determine local currency (prioritize position currency, fallback to cash currency, default GBP)
enriched = enriched.withColumn(
    "local_currency",
    F.upper(F.coalesce(
        F.nullif(F.trim(F.col("position_currency")), ""),
        F.nullif(F.trim(F.col("cash_currency")), ""),
        F.lit("GBP")
    ))
)

# Identify cash lines (no ISIN and no SEDOL)
enriched = enriched.withColumn(
    "_is_cash_line",
    (F.col("isin").isNull() | (F.trim(F.col("isin")) == "")) &
    (F.col("sedol").isNull() | (F.trim(F.col("sedol")) == ""))
)

# Apply identifier resolution logic
enriched = enriched.withColumn(
    "security_code",
    F.when(
        F.col("sedol").isNotNull() & (F.trim(F.col("sedol")) != ""),
        F.upper(F.trim(F.col("sedol")))
    )
    .when(
        F.col("isin").isNotNull() & (F.trim(F.col("isin")) != ""),
        F.upper(F.trim(F.col("isin")))
    )
    .when(
        F.col("_is_cash_line"),
        F.concat(F.lit("TPY_CASH_"), F.col("local_currency"))
    )
    .otherwise(None)
)

# Set ID type
enriched = enriched.withColumn(
    "id_type",
    F.when(
        F.col("sedol").isNotNull() & (F.trim(F.col("sedol")) != ""),
        F.lit("SEDOL")
    )
    .when(
        F.col("isin").isNotNull() & (F.trim(F.col("isin")) != ""),
        F.lit("ISIN")
    )
    .when(
        F.col("_is_cash_line"),
        F.lit("Undertaking - Specific")
    )
    .otherwise(None)
)

# Track which identifier was chosen
enriched = enriched.withColumn(
    "identifier_chosen",
    F.when(
        F.col("sedol").isNotNull() & (F.trim(F.col("sedol")) != ""),
        F.lit("SEDOL")
    )
    .when(
        F.col("isin").isNotNull() & (F.trim(F.col("isin")) != ""),
        F.lit("ISIN")
    )
    .when(
        F.col("_is_cash_line"),
        F.lit("SYNTHETIC_CASH")
    )
    .otherwise(F.lit("NONE"))
)

# Distribution of identifier types
identifier_dist = enriched.groupBy("identifier_chosen").count().orderBy(F.desc("count"))
print("  Identifier resolution distribution:")
for row in identifier_dist.collect():
    print(f"    {row['identifier_chosen']}: {row['count']} rows")

print("\n[STEP 5] Security identifier resolution complete")

## Step 6: Security Master Enrichment

Enrich asset names from security master, with fallback to source instrument name.

In [ ]:
# ========== Security Master Enrichment ==========
print("\n[STEP 6] Enriching from security master...")
print("=" * 70)

if security_master is not None:
    # Join on ISIN first
    enriched = enriched.join(
        security_master,
        F.upper(F.trim(enriched["isin"])) == security_master["isin_master"],
        "left"
    )
    
    # Fallback: if no ISIN match, try SEDOL
    # (would need another join, skip for now - prioritize ISIN)
    
    enriched = enriched.withColumn(
        "asset_name",
        F.when(
            F.col("_is_cash_line"),
            F.concat(F.lit("CASH - "), F.col("local_currency"))
        )
        .otherwise(
            F.upper(F.coalesce(
                F.col("asset_name_master"),
                F.col("instrument_name")
            ))
        )
    )
    
    enriched_count = enriched.filter(F.col("asset_name_master").isNotNull()).count()
    print(f"  Rows enriched from security master: {enriched_count}")
    print(f"  Rows using source instrument name: {combined_count - enriched_count}")
else:
    # No security master - use source instrument name
    enriched = enriched.withColumn(
        "asset_name",
        F.when(
            F.col("_is_cash_line"),
            F.concat(F.lit("CASH - "), F.col("local_currency"))
        )
        .otherwise(F.upper(F.col("instrument_name")))
    )
    print("  Using source instrument names (no security master)")

print("\n[STEP 6] Asset name enrichment complete")

## Step 6A: DFM Step 5 Lookup Validation

Binary validation equivalent to Excel Step 5 (`#N/A` checks). This identifies missing policy/security/asset mappings before value conversion and downstream checks.

In [ ]:
# ========== Step 6A: Lookup Validation (Excel Step 5 Equivalent) ==========
print("\n[STEP 6A] Checking lookup completeness (no #N/A equivalent)...")
print("=" * 70)

missing_policy_count = enriched.filter(F.col("policyholder_number").isNull() | (F.trim(F.col("policyholder_number")) == "")).count()
missing_identifier_count = enriched.filter(F.col("security_code").isNull() | (F.trim(F.col("security_code")) == "")).count()
missing_asset_name_count = enriched.filter(
    (~F.col("_is_cash_line")) & (F.col("asset_name").isNull() | (F.trim(F.col("asset_name")) == ""))
).count()

print(f"  Missing policy mappings: {missing_policy_count}")
print(f"  Missing security identifiers: {missing_identifier_count}")
print(f"  Missing asset names (non-cash): {missing_asset_name_count}")

if (missing_policy_count + missing_identifier_count + missing_asset_name_count) > 0:
    print("\n  [HITL REQUIRED] Lookup gaps detected. Review exception samples below.")

    print("\n  Sample missing policies:")
    enriched.filter(F.col("policyholder_number").isNull() | (F.trim(F.col("policyholder_number")) == "")) \
        .select("client_id", "source_file", "source_row_id") \
        .show(10, truncate=False)

    print("\n  Sample missing security identifiers:")
    enriched.filter(F.col("security_code").isNull() | (F.trim(F.col("security_code")) == "")) \
        .select("client_id", "instrument_name", "isin", "sedol", "source_row_id") \
        .show(10, truncate=False)

    print("\n  Sample missing asset names:")
    enriched.filter((~F.col("_is_cash_line")) & (F.col("asset_name").isNull() | (F.trim(F.col("asset_name")) == ""))) \
        .select("client_id", "security_code", "instrument_name", "source_row_id") \
        .show(10, truncate=False)
else:
    print("\n  [PASS] All key lookups populated.")

print("\n[STEP 6A] Lookup validation complete")

## Step 7: Convert Values and Apply FX

Parse numeric values, apply FX rates, calculate GBP equivalents.

In [ ]:
# ========== Value Conversion and FX Application ==========
print("\n[STEP 7] Converting values and applying FX rates...")
print("=" * 70)

# Helper: convert European decimal format "1.234,56" to "1234.56"
def parse_euro_decimal(col_name):
    return F.regexp_replace(
        F.regexp_replace(F.trim(F.col(col_name)), r"\.", ""),
        ",", "."
    ).cast("double")

# Parse position values
enriched = enriched.withColumn("bid_value_local", parse_euro_decimal("balance_local"))
enriched = enriched.withColumn("accrued_interest_local", parse_euro_decimal("accrued_interest_local"))

# Parse cash values
enriched = enriched.withColumn("cash_value_local", parse_euro_decimal("cash_balance_local"))

# Apply FX rates
if fx_rates is not None:
    enriched = enriched.join(
        fx_rates,
        enriched["local_currency"] == fx_rates["currency"],
        "left"
    )
    
    # For GBP, rate = 1.0
    enriched = enriched.withColumn(
        "fx_rate",
        F.when(F.col("local_currency") == "GBP", F.lit(1.0))
         .otherwise(F.col("fx_rate"))
    )
else:
    # No FX rates - assume everything is GBP
    enriched = enriched.withColumn("fx_rate", F.lit(1.0))

# Calculate GBP values
enriched = enriched.withColumn(
    "bid_value_gbp",
    F.when(F.col("bid_value_local").isNull(), F.lit(None).cast("double"))
     .when(F.col("fx_rate").isNull(), F.lit(None).cast("double"))
     .otherwise(F.col("bid_value_local") * F.col("fx_rate"))
)

enriched = enriched.withColumn(
    "cash_value_gbp",
    F.when(F.col("cash_value_local").isNull(), F.lit(0.0))
     .when(F.col("fx_rate").isNull(), F.lit(None).cast("double"))
     .otherwise(F.col("cash_value_local") * F.col("fx_rate"))
)

enriched = enriched.withColumn(
    "accrued_interest_gbp",
    F.when(F.col("accrued_interest_local").isNull(), F.lit(0.0))
     .when(F.col("fx_rate").isNull(), F.lit(None).cast("double"))
     .otherwise(F.col("accrued_interest_local") * F.col("fx_rate"))
)

# Holdings and price (not available in Brown Shipley source)
enriched = enriched.withColumn("holding", F.lit(None).cast("double"))
enriched = enriched.withColumn("local_bid_price", F.lit(None).cast("double"))

print("  Value conversion complete")
print("  FX rates applied")
print("  GBP equivalents calculated")

print("\n[STEP 7] Value conversion and FX application complete")

## Step 8: Include/Remove Flagging

Apply Brown Shipley workbook-equivalent decision table using policyholder number (`D`) and IH lookup output (`L`):
- Rule 1: IH match + IH status `Exclude` -> `Remove - unreliable values`
- Rule 2: blank policyholder number -> `Remove`
- Rule 3: literal `REMOVE*` policyholder marker -> `Remove`
- Rule 4: `L = 0` (no IH match) -> `Remove`
- Rule 5: IH match exists -> `Include`
- Rule 6: default -> `Remove`

For compatibility with downstream notebooks, both `include` (workbook label) and `include_flag` (pipeline field) are populated.

In [ ]:
# ========== Include/Remove Flagging ==========
print("\n[STEP 8] Applying Include/Remove logic...")
print("=" * 70)

# Workbook-equivalent rule fields
enriched = enriched.withColumn(
    "policyholder_number_clean",
    F.upper(F.trim(F.coalesce(F.col("policyholder_number"), F.lit(""))))
)

enriched = enriched.withColumn("ih_lookup_count", F.coalesce(F.col("ih_lookup_count"), F.lit(0)))
enriched = enriched.withColumn("ih_exclude_flag", F.coalesce(F.col("ih_exclude_flag"), F.lit(False)))
enriched = enriched.withColumn("ih_policy_exists", F.col("ih_lookup_count") > 0)

# Excel decision table precedence:
# 1) IH exists + Exclude -> Remove - unreliable values
# 2) Blank policy -> Remove
# 3) Literal REMOVE* marker -> Remove
# 4) L=0 (no IH match) -> Remove
# 5) IH exists -> Include
# 6) Default -> Remove
enriched = enriched.withColumn(
    "include",
    F.when(
        F.col("ih_policy_exists") & F.col("ih_exclude_flag"),
        F.lit("Remove - unreliable values")
    )
    .when(
        F.col("policyholder_number_clean") == "",
        F.lit("Remove")
    )
    .when(
        F.col("policyholder_number_clean") == "REMOVE*",
        F.lit("Remove")
    )
    .when(
        F.col("ih_lookup_count") == 0,
        F.lit("Remove")
    )
    .when(
        F.col("ih_policy_exists"),
        F.lit("Include")
    )
    .otherwise(F.lit("Remove"))
)

# Keep pipeline-compatible include flag
enriched = enriched.withColumn(
    "include_flag",
    F.when(F.col("include") == "Include", F.lit("Include")).otherwise(F.lit("Remove"))
)

# Reason codes for traceability
enriched = enriched.withColumn(
    "exclusion_reason_code",
    F.when(
        F.col("ih_policy_exists") & F.col("ih_exclude_flag"),
        F.lit("REMOVE_UNRELIABLE_IH_EXCLUDE")
    )
    .when(
        F.col("policyholder_number_clean") == "",
        F.lit("REMOVE_BLANK_POLICY")
    )
    .when(
        F.col("policyholder_number_clean") == "REMOVE*",
        F.lit("REMOVE_LITERAL_MARKER")
    )
    .when(
        F.col("ih_lookup_count") == 0,
        F.lit("REMOVE_NOT_IN_IH")
    )
    .otherwise(F.lit(None).cast("string"))
)

# Distributions
include_dist = enriched.groupBy("include").count().orderBy("include")
print("  Include/Remove distribution (workbook label):")
for row in include_dist.collect():
    print(f"    {row['include']}: {row['count']} rows")

pipeline_include_dist = enriched.groupBy("include_flag").count().orderBy("include_flag")
print("\n  include_flag distribution (pipeline):")
for row in pipeline_include_dist.collect():
    print(f"    {row['include_flag']}: {row['count']} rows")

exclusion_reasons = enriched.filter(F.col("include_flag") == "Remove") \
    .groupBy("exclusion_reason_code").count().orderBy(F.desc("count"))
if exclusion_reasons.count() > 0:
    print("\n  Exclusion reasons:")
    for row in exclusion_reasons.collect():
        print(f"    {row['exclusion_reason_code']}: {row['count']} rows")

print("\n[STEP 8] Include/Remove flagging complete")

## Step 9: Add Check Columns and Decision Trace

Add validation check flags and decision traceability.

In [ ]:
# ========== Check Columns and Decision Trace ==========
print("\n[STEP 9] Adding check columns and decision trace...")
print("=" * 70)

# DFM Step 7 check uses IH Report Movt% tolerance (98-102).
# Movt may arrive as '101%', '101.2', or null.
enriched = enriched.withColumn(
    "movt_percent",
    F.regexp_replace(F.trim(F.coalesce(F.col("movt_raw"), F.lit(""))), "%", "").cast("double")
)

# Movt-based holdings check (replaces unavailable holding x price check for this DFM).
enriched = enriched.withColumn(
    "holdings_check_flag",
    F.when(F.col("movt_percent").isNull(), F.lit("not_evaluable"))
     .when((F.col("movt_percent") >= 98.0) & (F.col("movt_percent") <= 102.0), F.lit("pass"))
     .otherwise(F.lit("fail"))
)

# Acquisition value check (stub - not available in source)
enriched = enriched.withColumn("acq_value_check_flag", F.lit("not_evaluable"))

# Decision trace JSON
enriched = enriched.withColumn(
    "decision_trace_json",
    F.to_json(
        F.struct(
            F.col("client_id").alias("policy_original"),
            F.col("policyholder_number").alias("policy_mapped"),
            F.col("policy_mapping_applied"),
            F.col("ih_policy_exists"),
            F.col("ih_exclude_flag"),
            F.col("ih_lookup_count"),
            F.col("include").alias("include_workbook"),
            F.col("include_flag").alias("include_pipeline"),
            F.col("identifier_chosen"),
            F.col("sedol").alias("source_sedol"),
            F.col("isin").alias("source_isin"),
            F.col("security_code"),
            F.col("exclusion_reason_code"),
            F.col("movt_percent"),
            F.col("holdings_check_flag")
        )
    )
)

# Data quality flags
enriched = enriched.withColumn(
    "data_quality_flags",
    F.array(
        F.when(F.col("fx_rate").isNull(), F.lit("FX_NOT_AVAILABLE")),
        F.when(F.col("policy_mapping_applied") == False, F.lit("POLICY_NOT_MAPPED")),
        F.when(F.col("movt_percent").isNull(), F.lit("MOVT_NOT_AVAILABLE")),
        F.when(F.col("holdings_check_flag") == "fail", F.lit("MOVT_OUTSIDE_TOLERANCE"))
    ).cast("array<string>")
)

# Remove nulls from array
enriched = enriched.withColumn(
    "data_quality_flags",
    F.expr("filter(data_quality_flags, x -> x is not null)")
)

check_dist = enriched.groupBy("holdings_check_flag").count().orderBy("holdings_check_flag")
print("  Movt tolerance check distribution:")
for row in check_dist.collect():
    print(f"    {row['holdings_check_flag']}: {row['count']} rows")

failed_movt = enriched.filter(F.col("holdings_check_flag") == "fail")
if failed_movt.count() > 0:
    print("\n  Sample Movt failures (outside 98-102):")
    failed_movt.select("client_id", "movt_raw", "movt_percent", "include_flag").show(10, truncate=False)

print("  Check columns added")
print("  Decision trace JSON generated")
print("  Data quality flags populated")

print("\n[STEP 9] Check columns and decision trace complete")

## Step 9A: Build HITL Review Queues

Creates review datasets for human-in-the-loop decisions:
- Step 6a: missing security mapping
- Step 6b: missing policy mapping
- Step 8: `Movt` anomalies requiring review

In [ ]:
# ========== Step 9A: Build HITL Review Queues ==========
print("\n[STEP 9A] Building HITL review datasets...")
print("=" * 70)

# Step 6a queue: missing security mapping context for manual enrichment.
queue_missing_security = enriched.filter(
    (~F.col("_is_cash_line")) &
    (
        F.col("security_code").isNull() |
        (F.trim(F.col("security_code")) == "") |
        F.col("asset_name").isNull() |
        (F.trim(F.col("asset_name")) == "")
    )
).select(
    "period", "run_id", "client_id", "instrument_name", "isin", "sedol", "security_code", "asset_name", "source_file", "source_row_id"
)

# Step 6b queue: missing policy mapping cases for manual policy resolution.
queue_missing_policy = enriched.filter(
    F.col("policyholder_number").isNull() | (F.trim(F.col("policyholder_number")) == "")
).select(
    "period", "run_id", "client_id", "instrument_name", "security_code", "movt_raw", "source_file", "source_row_id"
)

# Step 8 queue: Movt anomalies that need human review/escalation.
queue_movt_anomalies = enriched.filter(
    F.col("holdings_check_flag") == "fail"
).select(
    "period", "run_id", "client_id", "policyholder_number", "movt_raw", "movt_percent", "include_flag", "source_file", "source_row_id"
)

missing_security_count = queue_missing_security.count()
missing_policy_count = queue_missing_policy.count()
movt_anomaly_count = queue_movt_anomalies.count()

print(f"  Step 6a queue (missing security mapping): {missing_security_count}")
print(f"  Step 6b queue (missing policy mapping): {missing_policy_count}")
print(f"  Step 8 queue (Movt anomaly / policy review): {movt_anomaly_count}")

if missing_security_count > 0:
    print("\n  Sample Step 6a rows:")
    queue_missing_security.show(10, truncate=False)

if missing_policy_count > 0:
    print("\n  Sample Step 6b rows:")
    queue_missing_policy.show(10, truncate=False)

if movt_anomaly_count > 0:
    print("\n  Sample Step 8 rows:")
    queue_movt_anomalies.show(10, truncate=False)

# Optional persistence for operational review.
# Uncomment if you want durable queues:
# queue_missing_security.write.mode("overwrite").saveAsTable("stage2_review_queue_missing_security")
# queue_missing_policy.write.mode("overwrite").saveAsTable("stage2_review_queue_missing_policy")
# queue_movt_anomalies.write.mode("overwrite").saveAsTable("stage2_review_queue_movt_anomalies")

print("\n[STEP 9A] HITL review queues ready")

## Step 10: Project to Stage 2 Contract

Project to `individual_dfm_consolidated` schema.

In [ ]:
# ========== Project to Stage 2 Contract ==========
print("\n[STEP 10] Projecting to individual_dfm_consolidated schema...")
print("=" * 70)

# Generate row hash for deduplication
enriched = enriched.withColumn(
    "row_hash",
    F.sha2(
        F.concat_ws(
            "|",
            F.col("period"),
            F.col("dfm_id"),
            F.col("policyholder_number"),
            F.coalesce(F.col("security_code"), F.lit("")),
            F.coalesce(F.col("isin"), F.lit("")),
            F.coalesce(F.col("source_row_id"), F.lit(""))
        ),
        256
    )
)

# Parse value_date to date
enriched = enriched.withColumn(
    "report_date",
    F.coalesce(
        F.to_date(F.col("value_date"), "dd/MM/yyyy"),
        F.to_date(F.col("value_date"), "yyyy-MM-dd"),
        F.to_date(F.col("value_date"))
    )
)

# Select Stage 2 contract columns
stage2_output = enriched.select(
    # Provenance
    F.col("period"),
    F.col("run_id"),
    F.col("dfm_id"),
    F.lit(profile_id).alias("profile_id"),
    F.col("source_file"),
    F.col("source_row_id"),
    F.col("row_hash"),
    
    # Policy identifiers
    F.col("policyholder_number"),
    
    # Security identifiers
    F.col("security_code"),
    F.upper(F.trim(F.col("isin"))).alias("isin"),
    F.upper(F.trim(F.col("sedol"))).alias("sedol"),
    F.lit(None).cast("string").alias("other_security_id"),
    F.col("id_type"),
    F.col("asset_name"),
    
    # Position values
    F.col("holding").cast("decimal(28,10)"),
    F.col("local_bid_price").cast("decimal(28,10)"),
    F.col("local_currency"),
    F.col("fx_rate").cast("decimal(28,10)"),
    F.col("cash_value_gbp").cast("decimal(28,10)"),
    F.col("bid_value_gbp").cast("decimal(28,10)"),
    F.col("accrued_interest_gbp").cast("decimal(28,10)"),
    
    # Include/Remove
    F.col("include_flag"),
    F.col("exclusion_reason_code"),
    
    # Decision trace
    F.col("identifier_chosen"),
    F.col("decision_trace_json"),
    F.col("data_quality_flags"),
    
    # Metadata
    F.col("report_date"),
    F.current_timestamp().alias("transformed_at")
)

# Remove duplicates based on row_hash
stage2_output = stage2_output.dropDuplicates(["row_hash"])

output_count = stage2_output.count()
print(f"  Stage 2 output rows: {output_count}")
print(f"  Input rows (Stage 1): {combined_count}")

if output_count != combined_count:
    print(f"  [WARNING] Row count changed. Expected {combined_count}, got {output_count}")

print("\n[STEP 10] Schema projection complete")

## Step 11: Write to individual_dfm_consolidated

Persist Stage 2 output table.

In [ ]:
# ========== Write Stage 2 Output ==========
print("\n[STEP 11] Writing to individual_dfm_consolidated table...")
print("=" * 70)

try:
    stage2_output.write.mode("append").saveAsTable("individual_dfm_consolidated")
    print(f"  ✓ Successfully wrote {output_count} rows to individual_dfm_consolidated")
except Exception as e:
    print(f"  [ERROR] Failed to write to individual_dfm_consolidated: {e}")
    mssparkutils.notebook.exit("FAILED")

print("\n[STEP 11] Stage 2 output written successfully")

## Summary and Validation

Final statistics and validation checks.

In [ ]:
# ========== Summary ==========
print("\n" + "=" * 70)
print("STAGE 2 TRANSFORMATION COMPLETE")
print("=" * 70)

print(f"\nPeriod: {period}")
print(f"Run ID: {run_id}")
print(f"DFM: {dfm_name}")

print(f"\nRow Counts:")
print(f"  Stage 1 input (positions): {positions_count}")
if cash is not None:
    print(f"  Stage 1 input (cash): {cash_count}")
print(f"  Combined: {combined_count}")
print(f"  Stage 2 output: {output_count}")

include_summary = stage2_output.groupBy("include_flag").count().collect()
print(f"\nInclude/Remove Summary:")
for row in include_summary:
    print(f"  {row['include_flag']}: {row['count']} rows")

print(f"\nNext Steps:")
print(f"  1. Run nb_stage3_tpir_projection to create tpir_load_equivalent")
print(f"  2. Run nb_validate to generate dq_results and dq_exception_rows")
print(f"  3. Review validation results before publish")

print("\n" + "=" * 70)
mssparkutils.notebook.exit("OK")